# Image Classification using Convolutional Neural Networks (CNN)

**Author:** Harsh Raj  

**Registration Number:** 23MIM10089 

**Application Number:** IN26011390

**Batch Number:** 1A

**Email ID:** harsh.23mim10089@vitbhopal.ac.in

In [ ]:
import os
import getpass
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
# Task 1: Dataset Acquisition
!pip install -q kaggle

os.environ['KAGGLE_USERNAME'] = input("Kaggle Username: ")
os.environ['KAGGLE_KEY'] = getpass.getpass("Kaggle API Key: ")

from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()
api.dataset_download_files('bhavikjikadara/dog-and-cat-classification-dataset', path='.', unzip=True)

print("--- Folder Structure ---")
for root, dirs, files in os.walk('.'):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    depth = root.count(os.sep)
    if depth <= 2:
        img_count = len([f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"{'  ' * depth}{os.path.basename(root) or root}/  ({img_count} images)")

In [ ]:
# Task 1 (contd.): Dataset Exploration
from PIL import Image

base_dir = 'PetImages'
classes = ['Cat', 'Dog']

def list_images(folder):
    return [os.path.join(base_dir, folder, f) for f in os.listdir(os.path.join(base_dir, folder))
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

cat_paths = list_images('Cat')
dog_paths = list_images('Dog')

with Image.open(cat_paths[0]) as im:
    w, h = im.size
    ch = len(im.getbands())

print("--- Dataset Summary ---")
print(f"Classes         : {classes}")
print(f"Cat images      : {len(cat_paths)}")
print(f"Dog images      : {len(dog_paths)}")
print(f"Total images    : {len(cat_paths) + len(dog_paths)}")
print(f"Sample size     : {w}x{h}, {ch} channels\n")

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
sample_set = [(cat_paths[i], 'Cat') for i in range(3)] + [(dog_paths[i], 'Dog') for i in range(3)]
for ax, (path, label) in zip(axes, sample_set):
    with Image.open(path) as im:
        ax.imshow(im)
        ax.set_title(label)
        ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Preprocessing and Data Generators
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

records = []
for label in classes:
    folder = os.path.join(base_dir, label)
    for fname in os.listdir(folder):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            records.append({'filepath': os.path.join(folder, fname), 'label': label})

df = pd.DataFrame(records)

# Drop unreadable/corrupted files
keep = []
for _, row in df.iterrows():
    try:
        with Image.open(row['filepath']) as im:
            im.verify()
        keep.append(row)
    except Exception:
        continue

df_clean = pd.DataFrame(keep)
print(f"Usable images after cleaning: {len(df_clean)}")

train_df, test_df = train_test_split(
    df_clean, test_size=0.2, random_state=7, stratify=df_clean['label']
)
print(f"Train set: {len(train_df)}  |  Test set: {len(test_df)}")

datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_gen = datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='label',
    target_size=(128, 128), batch_size=32, class_mode='binary', shuffle=True
)

test_gen = datagen.flow_from_dataframe(
    test_df, x_col='filepath', y_col='label',
    target_size=(128, 128), batch_size=32, class_mode='binary', shuffle=False
)

In [ ]:
# Task 3: CNN Architecture
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.summary()

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    train_gen,
    epochs=10,
    validation_data=test_gen,
    verbose=1
)

In [ ]:
# Task 4: Model Evaluation
test_gen.reset()
probs = model.predict(test_gen, verbose=1)
y_pred = (probs > 0.5).astype(int).reshape(-1)
y_true = test_gen.classes
label_names = list(test_gen.class_indices.keys())

test_loss, test_acc = model.evaluate(test_gen, verbose=0)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("--- Performance Metrics ---")
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1-Score      : {f1:.4f}\n")

print("--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=label_names, yticklabels=label_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'], marker='o', label='Train')
axes[0].plot(history.history['val_accuracy'], marker='s', label='Validation')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.4, linestyle='--')

axes[1].plot(history.history['loss'], marker='o', label='Train')
axes[1].plot(history.history['val_loss'], marker='s', label='Validation')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.4, linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Task 5: Observations
print("--- Observations ---")
print(f"1. Test accuracy: {test_acc:.4f}, F1-score: {f1:.4f}.")
print(f"2. Confusion matrix breakdown:\n{cm}")
print("3. Compare training vs validation accuracy/loss curves above to check for overfitting;")
print("   if the gap widens, consider Dropout, data augmentation, or early stopping.")

## Conclusion

This notebook implements a Convolutional Neural Network (CNN) using Keras/TensorFlow to classify pet images into Cats and Dogs. The model uses three convolution-pooling blocks followed by a dense classification head with dropout regularization.

CNNs extract spatial features through learnable filters, and pooling layers reduce dimensionality while preserving important patterns like edges and textures. Compared to a plain fully connected network, a CNN uses far fewer parameters thanks to weight sharing across the image, while still capturing 2D spatial structure.

A known limitation of CNNs is their tendency to overfit on small or imbalanced datasets, and their sensitivity to geometric distortions unless data augmentation is applied — both worth watching for when interpreting the metrics above.